# Univariate EDA: Distributions, Outliers, and Review Notes

This notebook performs a focused univariate exploratory data analysis for variables marked as `conditional_review` or `needs_group_review` in `data/references/leakage_conceptual_screening.csv`.

The notebook includes: statistical summaries, missingness, outlier identification, categorical cardinality review, and a recommendation status for each reviewed feature.

In [ ]:
import os
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

ROOT = Path.cwd()
FIG_DIR = ROOT / 'notebooks' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH = ROOT / 'notebooks' / 'eda_univariate_summary.csv'

reference_dir = ROOT / 'data' / 'references'
leakage_file = reference_dir / 'leakage_conceptual_screening.csv'
schema_file = reference_dir / 'silver_schema_data_dictionary.csv'
leakage_df = pd.read_csv(leakage_file)
schema_df = pd.read_csv(schema_file)

review_vars = sorted(leakage_df.loc[leakage_df['screening_status'].isin(['needs_group_review', 'conditional_review']), 'variable_name'].unique())
review_schema = schema_df[schema_df['silver_column_name'].isin(review_vars)].copy()

print(f'Review variables identified: {len(review_vars)}')
print(review_schema[['silver_column_name', 'silver_data_type', 'review_status', 'leakage_restriction']].head(25))

In [ ]:
CANONICAL_FILENAME = 'DataCoSupplyChainDataset.csv'
candidate_path = ROOT / 'data' / 'bronze' / 'dataco' / CANONICAL_FILENAME
if not candidate_path.exists():
    env_path = os.environ.get('DATASET_PATH')
    if env_path:
        candidate_path = Path(env_path)

if not candidate_path.exists():
    candidate_folder = ROOT / 'data' / 'bronze' / 'dataco'
    candidates = []
    if candidate_folder.exists():
        candidates = sorted([p for p in candidate_folder.glob('*.csv') if p.is_file()])
    if candidates:
        print('Candidate CSV files found in data/bronze/dataco/:')
        for c in candidates:
            print('-', c.name)
    raise FileNotFoundError(
        f'Dataset not found at {candidate_path}. Place DataCoSupplyChainDataset.csv in data/bronze/dataco/ or set DATASET_PATH to the CSV path.'
    )

dataset_file = candidate_path
print('Using dataset file:', dataset_file)
df = pd.read_csv(dataset_file, low_memory=False)
print('Loaded rows:', len(df), 'columns:', len(df.columns))
print('Dataset column sample:')
print(df.columns.to_list()[:40])

In [ ]:
def normalize_column_name(name):
    text = str(name).strip().lower()
    text = re.sub(r'[\s_\-()]+', ' ', text)
    text = re.sub(r'[^a-z0-9 ]+', '', text)
    return text

column_lookup = {normalize_column_name(col): col for col in df.columns}

def resolve_column(raw_name):
    normalized = normalize_column_name(raw_name)
    if normalized in column_lookup:
        return column_lookup[normalized]
    substring_matches = [target for target in column_lookup if normalized in target or target in normalized]
    if len(substring_matches) == 1:
        return column_lookup[substring_matches[0]]
    return None

matched = []
unmatched = []
for variable_name in review_vars:
    resolved = resolve_column(variable_name)
    if resolved is not None:
        matched.append((variable_name, resolved))
    else:
        unmatched.append(variable_name)

print(f'Variables matched to dataset columns: {len(matched)}')
print(f'Variables unmatched: {len(unmatched)}')
if unmatched:
    for var in unmatched:
        print('-', var)

In [ ]:
def infer_datetime(series):
    if pd.api.types.is_datetime64_any_dtype(series):
        return True
    coerced = pd.to_datetime(series.dropna().unique(), errors='coerce')
    return coerced.notna().mean() >= 0.9

def safe_filename(name):
    cleaned = re.sub(r'[^A-Za-z0-9_\-]+', '_', name)
    return cleaned.strip('_').lower()

def numeric_outlier_stats(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier_mask = series.lt(lower) | series.gt(upper)
    return {
        'q1': q1,
        'q3': q3,
        'iqr': iqr,
        'lower_fence': lower,
        'upper_fence': upper,
        'outlier_count': int(outlier_mask.sum()),
        'outlier_rate': float(outlier_mask.mean())
    }

summary_rows = []
for variable_name in review_vars:
    dataset_col = resolve_column(variable_name)
    if dataset_col is None:
        summary_rows.append({
            'variable_name': variable_name,
            'dataset_column': None,
            'status': 'missing_in_dataset',
            'missing_rate': None,
            'unique_values': None,
            'decision': 'needs_group_review',
            'notes': 'Review variable not found in the dataset. Confirm the source schema or column naming.',
        })
        continue

    series = df[dataset_col]
    missing_rate = float(series.isna().mean())
    unique_values = int(series.nunique(dropna=True))
    notes = []
    figure_path = FIG_DIR / f"{safe_filename(dataset_col)}.png"
    decision = 'needs_group_review'

    if infer_datetime(series):
        dt_series = pd.to_datetime(series, errors='coerce')
        missing_rate = float(dt_series.isna().mean())
        date_counts = dt_series.dt.to_period('M').value_counts().sort_index()
        notes.append('Datetime-like field; reviewed by month.')
        if missing_rate <= 0.1:
            decision = 'approved_for_gold'

        fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)
        date_counts.plot(kind='bar', ax=ax, color='tab:green')
        ax.set_title(f'Date distribution: {dataset_col}')
        ax.set_xlabel('Period')
        ax.set_ylabel('Count')
        fig.savefig(figure_path)
        plt.close(fig)

        summary_rows.append({
            'variable_name': variable_name,
            'dataset_column': dataset_col,
            'status': 'datetime',
            'missing_rate': missing_rate,
            'unique_values': unique_values,
            'decision': decision,
            'notes': ' '.join(notes),
            'summary_min': dt_series.min(),
            'summary_q1': None,
            'summary_median': None,
            'summary_q3': None,
            'summary_max': dt_series.max(),
            'outlier_count': None,
        })
        continue

    numeric_series = pd.to_numeric(series, errors='coerce')
    numeric_ratio = float(numeric_series.notna().mean())
    if pd.api.types.is_numeric_dtype(series) or numeric_ratio >= 0.8:
        numeric_series = numeric_series.dropna()
        stats = numeric_series.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).to_dict()
        outlier_info = numeric_outlier_stats(numeric_series) if not numeric_series.empty else {'outlier_count': 0, 'outlier_rate': 0.0}
        notes.append(f'Numeric field, {unique_values} distinct non-null values.')
        notes.append(f'Outlier rate by IQR: {outlier_info["outlier_rate"]:.2%} ({outlier_info["outlier_count"]} outliers).')
        if outlier_info['outlier_rate'] > 0.05:
            notes.append('High outlier rate; review for data quality, transformation, or leakage.')
        if missing_rate <= 0.1 and outlier_info['outlier_rate'] <= 0.05:
            decision = 'approved_for_gold'

        fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 4), constrained_layout=True)
        if not numeric_series.empty:
            sns.histplot(numeric_series, ax=axes[0], bins=40, color='tab:blue')
            axes[0].set_title(f'Histogram: {dataset_col}')
            sns.boxplot(x=numeric_series, ax=axes[1], color='tab:orange')
            axes[1].set_title(f'Boxplot: {dataset_col}')
        else:
            axes[0].text(0.5, 0.5, 'No numeric data', ha='center', va='center')
            axes[1].text(0.5, 0.5, 'No numeric data', ha='center', va='center')
        fig.savefig(figure_path)
        plt.close(fig)

        summary_rows.append({
            'variable_name': variable_name,
            'dataset_column': dataset_col,
            'status': 'numeric',
            'missing_rate': missing_rate,
            'unique_values': unique_values,
            'decision': decision,
            'notes': ' '.join(notes),
            'summary_min': stats.get('min'),
            'summary_q1': stats.get('25%'),
            'summary_median': stats.get('50%'),
            'summary_q3': stats.get('75%'),
            'summary_max': stats.get('max'),
            'outlier_count': outlier_info['outlier_count']
        })
        continue

    categorical = series.fillna('NULL').astype(str)
    top_categories = categorical.value_counts(dropna=False).head(20)
    cardinality = unique_values
    notes.append(f'Categorical field with cardinality {cardinality}.')
    if cardinality > 100:
        notes.append('High cardinality; consider grouping, hashing, or train-only encoding.')
    if missing_rate > 0.1:
        notes.append('High missingness; review imputation or exclusion strategy.')
    if cardinality <= 100 and missing_rate <= 0.1:
        decision = 'approved_for_gold'

    fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
    sns.barplot(x=top_categories.values, y=top_categories.index, ax=ax, palette='crest')
    ax.set_title(f'Top categories: {dataset_col}')
    ax.set_xlabel('Count')
    fig.savefig(figure_path)
    plt.close(fig)

    summary_rows.append({
        'variable_name': variable_name,
        'dataset_column': dataset_col,
        'status': 'categorical',
        'missing_rate': missing_rate,
        'unique_values': cardinality,
        'decision': decision,
        'notes': ' '.join(notes),
        'summary_min': None,
        'summary_q1': None,
        'summary_median': None,
        'summary_q3': None,
        'summary_max': None,
        'outlier_count': None,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_PATH, index=False)
print('Saved summary to', SUMMARY_PATH)
print('Saved figures to', FIG_DIR)
print(summary_df.head(25).to_string(index=False))

In [ ]:
print('Decision counts:')
print(summary_df['decision'].value_counts(dropna=False))

print('')
print('Top review notes:')
print(summary_df[['variable_name', 'dataset_column', 'decision', 'missing_rate', 'unique_values', 'notes']].head(20).to_string(index=False))

## Findings Summary

- **Patterns:** Describe skewness, modal behavior, cardinality issues, and missingness trends observed in the review set.
- **Anomalies:** Note extreme outliers, unusual value ranges, missing-date gaps, or unexpected category counts.
- **Caveats:** Document limitations such as dataset availability, unknown schema mappings, and any columns that could not be matched.
- **Modeling implications:** Link each `approved_for_gold` or `needs_group_review` decision to model risk, target leakage, or feature engineering strategy.
- **Next steps:** Validate the summary with a teammate, confirm the review variable mappings, and decide on grouping/encoding for high-cardinality fields.